In [44]:
import digitalhub as dh

project = dh.get_or_create_project("test-xinet")

In [45]:
func = project.new_function(name="xinet-pose",
                            kind="openinference",
                            python_version="PYTHON3_13",
                            code_src="xinet",
                            handler="xinet_handler:handler",
                            init_function="init_context",
                            model_name="XiNet-s-pose-224",
                            inputs=[{
                                "name": "image",
                                "datatype": "UINT8",
                                "shape": [-1,-1]
                            }],
                            outputs=[{
                                "name": "image",
                                "datatype": "UINT8",
                                "shape": [-1,-1]
                            }],
                            requirements=["opencv-python==4.13.0.92", "numpy==2.4.3", "onnxruntime==1.24.3", "matplotlib==3.10.8"]
                           )

In [ ]:
build = func.run(
    "build",
    instructions=[
        "apt-get update && apt-get install -y libgl1 libglib2.0-0 libsm6 libxext6 libxrender1 libfontconfig1 libice6"
    ],
    wait=True
)

In [47]:
run = func.run(action="serve", resources={"mem": "1Gi", "disk": "5Gi"})

In [ ]:
import requests
import urllib.request

BASE_URL = run.refresh().status.service['url']
MODEL_NAME = 'XiNet-s-pose-224'

img_url = 'https://dn721803.ca.archive.org/0/items/2_20200617_202006/1.JPG'
with urllib.request.urlopen(img_url) as response:
    image_bytes = response.read()

    request = {
        "inputs": [
                {
                    "name": "input",
                    "datatype": "UINT8",
                    "shape": [1, len(image_bytes)],
                    "data": list(image_bytes)
                }
            ]    
    }

In [63]:
response = requests.post(
        f"http://{BASE_URL}/v2/models/{MODEL_NAME}/infer",
        json=request,
        headers={"Content-Type": "application/json"}
    )

print(f"Status Code: {response.status_code}")
data = response.json()

Status Code: 200


In [64]:
import time
from io import BytesIO
from PIL import Image

image_bytes = data['outputs'][0]['data']

timestamp = time.strftime("%Y%m%d-%H%M%S", time.gmtime())   
output_filename = f"./output/image_{timestamp}.jpg"

# Convert bytes to PIL Image
image_bytes_clean = bytes(image_bytes)
image = Image.open(BytesIO(image_bytes_clean)).convert('RGB')
image.save(output_filename)
